In [1]:
import pandas as pd

# Load the candidate table you already uploaded
stations_df = pd.read_csv("Research_Dataset-candidate_station_table.csv")


In [2]:
# Apply our Task #1 screening filters
filtered_stations = stations_df[
    (stations_df['number_of_usable_years'] >= 30) &
    (stations_df['percent_missing_summer_TMAX'] <= 10) &
    (stations_df['percent_missing_summer_TMIN'] <= 10) &
    (stations_df['elevation'] != -999.9)
]

# Create a set of valid station IDs for lightning-fast lookups
valid_station_ids = set(filtered_stations['station_id'])
print(f"Tracking {len(valid_station_ids)} high-quality stations.")

Tracking 369 high-quality stations.


In [ ]:
filtered_stations.head()

In [4]:
import numpy as np

big_file_path = "ca_daily_tmax_tmin_tavg.csv"  # <-- Change this to your actual filename
output_file_path = "clean_daily_temperatures.csv"

# Initialize a flag to handle the CSV header properly
is_first_chunk = True

# Process the massive file in chunks of 100,000 rows at a time
for chunk in pd.read_csv(big_file_path, chunksize=100000, low_memory=False):

    # 1. Filter ONLY include the high-quality candidate stations
    chunk = chunk[chunk['station_id'].isin(valid_station_ids)].copy()

    if chunk.empty:
        continue

    # 2. Restrict date ranges to the study period (1970-present)
    # Assumes your date column is named 'date' or 'year'. If it's a full date string, we extract the year:
    chunk['date'] = pd.to_datetime(chunk['date'])
    chunk = chunk[chunk['date'].dt.year >= 1970]

    if chunk.empty:
        continue

    # 3. Replace missing TAVG_C with the average of TMAX and TMIN
    # First, make sure TMAX and TMIN are numeric
    chunk['TMAX'] = pd.to_numeric(chunk['TMAX'], errors='coerce')
    chunk['TMIN'] = pd.to_numeric(chunk['TMIN'], errors='coerce')

    # Calculate the mid-point average
    calculated_tavg = (chunk['TMAX'] + chunk['TMIN']) / 2

    # Fill in the NaNs in TAVG_C using the calculated average
    if 'TAVG_C' in chunk.columns:
        chunk['TAVG_C'] = chunk['TAVG_C'].fillna(calculated_tavg)
    else:
        chunk['TAVG_C'] = calculated_tavg

    # 4. Save/Append the clean chunk to our final clean file
    if is_first_chunk:
        chunk.to_csv(output_file_path, mode='w', index=False)
        is_first_chunk = False
    else:
        chunk.to_csv(output_file_path, mode='a', header=False, index=False)

print("Finished processing! Your clean, compressed dataset is saved as 'clean_daily_temperatures.csv'.")

Finished processing! Your clean, compressed dataset is saved as 'clean_daily_temperatures.csv'.
